In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime
from pyspark.sql import Window

spark = SparkSession.builder \
    .master("local[2]") \
    .appName("TennisPipeline") \
    .getOrCreate()

In [59]:
file_to_read = '../data/bronze/atp_matches_1968.csv'
file_to_write = "../data/silver/atp_matches_1968.parquet"

df = spark.read \
.option('header', True) \
.option('inferSchema', True) \
.csv(file_to_read)

df = df.withColumn(
    'tourney_date',
    F.to_date(F.col('tourney_date'), 'yyyyMMdd')
)

df = df.withColumnsRenamed({
    'tourney_id': 'tournament_id',
    'tourney_name': 'tournament_name',
    'tourney_date': 'tournament_date',
    'match_num': 'match_number',
    'winner_ht': 'winner_height',
    'winner_ioc': 'winner_country_code',
    'loser_ht': 'loser_height',
    'loser_ioc': 'loser_country_code',
})

df = df.withColumn(
    'match_id',
    F.concat(
        F.col('tournament_id'),
        F.lit('_'),
        F.col('match_number')
    )
)

df = df.withColumn(
    'pipeline_run_id',
    F.lit(datetime.now().strftime('%Y%m%d_%H%M%S'))
)

window = Window.partitionBy('match_id').orderBy('match_number')

duplicates = df.withColumn('cnt', F.count('*').over(window)) \
    .filter(F.col('cnt') > 1) \
    .drop('cnt') \
    .count()
print(f"Duplicated rows in the DataFrame: {duplicates}")

df = df.withColumn('rank', F.row_number().over(window)) \
       .filter(F.col('rank') == 1) \
       .drop('rank')


critical_cols = ['tournament_id', 'tournament_name', 'surface', 
                 'tournament_date', 'winner_id', 'winner_name',
                 'loser_id', 'loser_name', 'score', 'round', 'best_of']

for col in critical_cols:
    null_count = df.filter(F.col(col).isNull()).count()
    if null_count > 0:
        print(f"WARNING: {col} has {null_count} nulls")

df.write.mode('overwrite').parquet(file_to_write)
df.show()

Duplicated rows in the DataFrame: 0
+-------------+---------------+-------+---------+-------------+---------------+------------+---------+-----------+------------+---------------+-----------+-------------+-------------------+----------+--------+----------+-----------+----------------+----------+------------+------------------+---------+------------+-------+-----+-------+-----+----+------+-------+--------+--------+-------+---------+---------+-----+----+------+-------+--------+--------+-------+---------+---------+-----------+------------------+----------+-----------------+-------------+---------------+
|tournament_id|tournament_name|surface|draw_size|tourney_level|tournament_date|match_number|winner_id|winner_seed|winner_entry|    winner_name|winner_hand|winner_height|winner_country_code|winner_age|loser_id|loser_seed|loser_entry|      loser_name|loser_hand|loser_height|loser_country_code|loser_age|       score|best_of|round|minutes|w_ace|w_df|w_svpt|w_1stIn|w_1stWon|w_2ndWon|w_SvGms|w_b

3076